### Data Cleaning v01

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/variant_summary.txt', sep='\t', low_memory=False)
df.head()

In [ ]:
print(df.shape)
print(df['ClinicalSignificance'].value_counts())

In [ ]:
# there are many ambiguous labels based on the ClinicalSignificance value counts
# resolution is to exclude those from the used data
labels_to_include = ['Pathogenic', 'Likely pathogenic', 'Benign', 'Likely benign']
df = df[df['ClinicalSignificance'].isin(labels_to_include)]

In [ ]:
# focusing on simple nucleotide variant for now
df = df[df['Type'] == 'single nucleotide variant']

In [ ]:
print(df['ReviewStatus'].value_counts())

# filter for high-quality review status only
high_quality = [
    'practice guideline',
    'reviewed by expert panel',
    'criteria provided, multiple submitters, no conflicts'
]
df = df[df['ReviewStatus'].isin(high_quality)]

In [ ]:
# check for missing data in columns of interest
important_cols = ['GeneSymbol', 'ClinicalSignificance', 'Assembly', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']
print(df[important_cols].isnull().sum())

In [ ]:
# check genome builds
print(df['Assembly'].value_counts())

# pick one assembly to avoid duplicates
# GRCh38 is the current reference
df = df[df['Assembly'] == 'GRCh38']

In [ ]:
# check whether #AlleleID deduplication actually removes anything
n_before = len(df)
n_after = df['#AlleleID'].nunique()
print(f'Rows before dedup: {n_before:,}')
print(f'Unique AlleleIDs:   {n_after:,}')
print(f'Rows removed:       {n_before - n_after:,}')

# check VariationID as alternative key
print(f'\nUnique VariationIDs: {df["VariationID"].nunique():,}')

# if rows are removed, inspect them
dupes = df[df.duplicated(subset=['#AlleleID'], keep=False)]
if len(dupes) > 0:
    print(f'\nDuplicate AlleleID rows ({len(dupes):,}):')
    print(dupes[['#AlleleID', 'VariationID', 'ClinicalSignificance']].sort_values('#AlleleID').head(10))
else:
    print('\nNo duplicate AlleleIDs found — dedup step has no effect.')

# drop duplicate variants
df = df.drop_duplicates(subset=['#AlleleID'], keep='first')

In [ ]:
print(df['Chromosome'].value_counts())

# remove non-standard instances for the sake of simplicity
# keep 1-22, X, and Y
standard = [str(i) for i in range(1, 23)] + ['X', 'Y']
df = df[df['Chromosome'].isin(standard)]

# drop rows where allele is missing (ClinVar uses 'na' as a placeholder)
df = df[(df['ReferenceAlleleVCF'] != 'na') & (df['AlternateAlleleVCF'] != 'na')]

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
# filter out relevant features
features_to_keep = [
    'GeneSymbol',
    'Chromosome',
    'Start',
    'Stop',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
]

# separate into target and feature tensors
label_map = {
    'Pathogenic': 1, 'Likely pathogenic': 1,
    'Benign': 0, 'Likely benign': 0
}
y = df['ClinicalSignificance'].map(label_map).rename('label')
X = df[features_to_keep].copy()
print('Features:')
print(X.head())
print('\nMissing values:')
print(X.isnull().sum())
print('\nTarget:')
print(y.head())
print('\nClass distribution:')
print(y.value_counts())

In [ ]:
# convert to JSON and save
y.to_frame().to_json('./data/target.json', orient='records')
X.to_json('./data/features.json', orient='records')